# PCA and dimensionality reduction

Reduce correlated features while keeping preprocessing inside cross-validation. Use scikit-learn's bundled 8×8 digits; this is different from MNIST and needs no download.

PCA chooses directions of variance without labels. High retained variance does not guarantee optimal classification.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target,
    test_size=0.2, stratify=data.target, random_state=42)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
models = {
    'scaled logistic regression': Pipeline([('scale', StandardScaler()),
        ('classify', LogisticRegression(max_iter=600))]),
    'PCA then logistic regression': Pipeline([('scale', StandardScaler()),
        ('reduce', PCA(n_components=0.95, svd_solver='full')),
        ('classify', LogisticRegression(max_iter=600))]),
}


## 1. Select from training-fold scores

The scaler and PCA are refitted in every fold. Fitting either on the full dataset before cross-validation would leak information from the validation data.


In [ ]:
rows = []
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=1)
    rows.append({'model': name, 'cv_accuracy': scores.mean(), 'cv_std': scores.std()})
comparison = pd.DataFrame(rows).set_index('model')
display(comparison.round(3))
chosen_name = comparison['cv_accuracy'].idxmax()
chosen = models[chosen_name].fit(X_train, y_train)
print('Selected:', chosen_name)
print('Held-out accuracy:', round(accuracy_score(y_test, chosen.predict(X_test)), 3))


## 2. Inspect PCA's training representation

The two-dimensional plot is a separate visualization fitted on training data only. Its labels color the plot after fitting; they do not influence PCA. This is not a t-SNE or a supervised embedding.


In [ ]:
pca_pipeline = models['PCA then logistic regression'].fit(X_train, y_train)
pca = pca_pipeline.named_steps['reduce']
print('Retained components:', pca.n_components_, 'of', X_train.shape[1])
print('Retained training variance:', round(pca.explained_variance_ratio_.sum(), 3))
visual = Pipeline([('scale', StandardScaler()), ('reduce', PCA(n_components=2, svd_solver='full'))])
embedding = visual.fit_transform(X_train)
fig, ax = plt.subplots(figsize=(6, 4))
plot = ax.scatter(embedding[:, 0], embedding[:, 1], c=y_train, s=9, cmap='tab10')
ax.set(xlabel='Principal component 1', ylabel='Principal component 2')
fig.colorbar(plot, ax=ax, ticks=range(10))
fig.tight_layout()
plt.show()


## Practice

Compare retained-variance thresholds using training-only validation. Inspect how compression changes fit time and validation accuracy. Keep test performance separate from exploratory plot interpretations.

**Reference:** [scikit-learn: PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca).
